In [1]:
import pandas as pd

In [2]:
# sec 01

In [3]:
train = pd.read_csv('flight_train.csv')
test  = pd.read_csv('flight_test.csv')
print(train.shape, test.shape)

(10505, 11) (4502, 10)


In [4]:
set(train.columns) - set(test.columns)

{'price'}

In [5]:
target = train.pop('price')
print(train.shape, test.shape, target.shape)

(10505, 10) (4502, 10) (10505,)


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10505 entries, 0 to 10504
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   airline           10505 non-null  object 
 1   flight            10505 non-null  object 
 2   source_city       10505 non-null  object 
 3   departure_time    10505 non-null  object 
 4   stops             10505 non-null  object 
 5   arrival_time      10505 non-null  object 
 6   destination_city  10505 non-null  object 
 7   class             10505 non-null  object 
 8   duration          10505 non-null  float64
 9   days_left         10505 non-null  int64  
dtypes: float64(1), int64(1), object(8)
memory usage: 820.8+ KB


In [7]:
train.head()

,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left
0,Vistara,UK-776,Kolkata,Evening,one,Late_Night,Delhi,Economy,6.58,31
1,Vistara,UK-852,Bangalore,Morning,zero,Morning,Mumbai,Business,1.92,37
2,Indigo,6E-2348,Delhi,Evening,one,Late_Night,Bangalore,Economy,5.58,25
3,Air_India,AI-763,Kolkata,Early_Morning,one,Evening,Chennai,Business,12.00,15
4,Indigo,6E-752,Hyderabad,Early_Morning,one,Evening,Chennai,Economy,9.50,20


In [8]:
cols = train.select_dtypes('object').columns
cols

Index(['airline', 'flight', 'source_city', 'departure_time', 'stops',
       'arrival_time', 'destination_city', 'class'],
      dtype='object')

In [9]:
train = train.drop(columns=['flight'])
test = test.drop(columns=['flight'])
print(train.shape, test.shape, target.shape)

(10505, 9) (4502, 9) (10505,)


In [10]:
train = pd.get_dummies(train)
test = pd.get_dummies(test)
print(train.shape, test.shape, target.shape)

(10505, 37) (4502, 37) (10505,)


In [11]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

In [12]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)

In [13]:
from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_pred, y_val)
print(f'rmse : {rmse:.3f}')

rmse : 4353.154


In [14]:
pred = rf.predict(test)
submit = pd.DataFrame({'pred':pred})
submit.to_csv('result1.csv')

In [15]:
# sec 02

In [16]:
train = pd.read_csv('laptop_train.csv')
test = pd.read_csv('laptop_test.csv')

In [17]:
train.shape, test.shape

((91, 10), (39, 9))

In [18]:
set(train.columns) - set(test.columns)

{'Price'}

In [19]:
target = train.pop('Price')
print(train.shape, test.shape, target.shape)

(91, 9) (39, 9) (91,)


In [20]:
train.isnull().sum()

Brand                  0
Model                  9
Series                36
Processor              5
Processor_Gen          5
RAM                    6
Hard_Disk_Capacity     6
OS                     6
Rating                 0
dtype: int64

In [21]:
train = train.fillna('X')
test = test.fillna('X')

In [22]:
train.isnull().sum()

Brand                 0
Model                 0
Series                0
Processor             0
Processor_Gen         0
RAM                   0
Hard_Disk_Capacity    0
OS                    0
Rating                0
dtype: int64

In [23]:
print(train.shape, test.shape)
merged = pd.concat([train, test], axis=0)
print(merged.shape)
merged = pd.get_dummies(merged)
print(merged.shape)
train = merged[:len(train)]
test = merged[len(train):]
print(train.shape, test.shape)

(91, 9) (39, 9)
(130, 9)
(130, 122)
(91, 122) (39, 122)


In [24]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.2, random_state=0)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

(72, 122) (19, 122) (72,) (19,)


In [25]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)

In [26]:
from sklearn.metrics import r2_score
r2 = r2_score(y_val, y_pred)
print(f'r2 : {r2:.3f}')

r2 : 0.762


In [27]:
# sec 03

In [28]:
train = pd.read_csv('car_train.csv')
test = pd.read_csv('car_test.csv')
print(train.shape, test.shape)

(6732, 17) (5772, 16)


In [29]:
set(train.columns) - set(test.columns)

{'Price'}

In [30]:
target = train.pop('Price')
print(train.shape, test.shape, target.shape)

(6732, 16) (5772, 16) (6732,)


In [31]:
for col in train.columns:
    print(col, train[col].nunique())

Levy 395
Manufacturer 55
Model 864
Prod. year 43
Category 11
Leather interior 2
Fuel type 6
Engine volume 91
Mileage 3430
Cylinders 11
Gear box type 4
Drive wheels 3
Doors 3
Wheel 2
Color 16
Airbags 16


In [32]:
train['Levy'] = train['Levy'].str.replace('-','0').astype(int)
train['is_Turbo'] = train['Engine volume'].str.endswith(' Turbo')
train['Engine volume'] = train['Engine volume'].str.rstrip(' Turbo').astype(float)
train['Mileage'] = train['Mileage'].str.rstrip(' km').astype(int)

test['Levy'] = test['Levy'].str.replace('-','0').astype(int)
test['is_Turbo'] = test['Engine volume'].str.endswith(' Turbo')
test['Engine volume'] =test['Engine volume'].str.rstrip(' Turbo').astype(float)
test['Mileage'] = test['Mileage'].str.rstrip(' km').astype(int)

In [33]:
print(train.shape, test.shape, target.shape)
train = train.drop(columns=['Model'])
test = test.drop(columns=['Model'])
print(train.shape, test.shape, target.shape)

(6732, 17) (5772, 17) (6732,)
(6732, 16) (5772, 16) (6732,)


In [34]:
train.isnull().sum()

Levy                0
Manufacturer        0
Prod. year          0
Category            0
Leather interior    0
Fuel type           0
Engine volume       0
Mileage             0
Cylinders           0
Gear box type       0
Drive wheels        0
Doors               0
Wheel               0
Color               0
Airbags             0
is_Turbo            0
dtype: int64

In [35]:
for col in train.select_dtypes('object').columns:
    print(col)
    print(train[col].nunique())
    print()

Manufacturer
55

Category
11

Leather interior
2

Fuel type
6

Gear box type
4

Drive wheels
3

Doors
3

Wheel
2

Color
16



In [36]:
print(train.shape, test.shape, target.shape)
train = pd.get_dummies(train)
test = pd.get_dummies(test)
print(train.shape, test.shape, target.shape)

(6732, 16) (5772, 16) (6732,)
(6732, 109) (5772, 109) (6732,)


In [39]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.2, random_state=0)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

(5385, 109) (1347, 109) (5385,) (1347,)


In [40]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)

In [41]:
from sklearn.metrics import root_mean_squared_log_error
rmsle = root_mean_squared_log_error(y_val, y_pred)
print(f'rmsle : {rmsle:.3f}')

rmsle : 1.060
